In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import cvxpy as cp


In [5]:
returns = pd.read_csv(

    "../data/processed/returns.csv",

    index_col=0,

    parse_dates=True

)

regimes = pd.read_csv(

    "../data/processed/regimes.csv",

    index_col=0,

    parse_dates=True

)

regimes.columns = ["Regime"]

In [6]:
returns = returns.loc[regimes.index]

In [7]:
def optimize_portfolio(

    expected_returns,

    covariance_matrix,

    risk_aversion=0.5

):

    n = len(expected_returns)

    w = cp.Variable(n)

    portfolio_return = expected_returns @ w

    portfolio_risk = cp.quad_form(

        w,

        covariance_matrix

    )

    objective = cp.Maximize(

        portfolio_return

        -

        risk_aversion * portfolio_risk

    )

    constraints = [

        cp.sum(w) == 1,

        w >= 0

    ]

    problem = cp.Problem(

        objective,

        constraints

    )

    problem.solve()

    return np.array(w.value).flatten()

In [8]:
optimized_allocations = {}

for regime in regimes["Regime"].unique():

    mask = regimes["Regime"] == regime

    regime_returns = returns.loc[mask]

    mu = regime_returns.mean() * 252

    cov = regime_returns.cov() * 252

    weights = optimize_portfolio(

        mu.values,

        cov.values

    )

    optimized_allocations[regime] = weights

    print()

    print("=" * 50)

    print(f"REGIME {regime}")

    print("=" * 50)

    allocation_df = pd.DataFrame({

        "Asset": returns.columns,

        "Weight": weights

    })

    print(allocation_df)


REGIME 0
           Asset        Weight
0    GOLDBEES.NS  1.323229e-02
1  LIQUIDBEES.NS  1.112208e-22
2       ^NSEBANK  9.867677e-01
3          ^NSEI  5.558030e-23

REGIME 2
           Asset        Weight
0    GOLDBEES.NS  9.978105e-03
1  LIQUIDBEES.NS  9.900219e-01
2       ^NSEBANK  2.249684e-26
3          ^NSEI  5.551041e-23

REGIME 1
           Asset        Weight
0    GOLDBEES.NS  1.000000e+00
1  LIQUIDBEES.NS -2.220484e-22
2       ^NSEBANK -1.108876e-22
3          ^NSEI -1.109051e-22

REGIME 3
           Asset        Weight
0    GOLDBEES.NS -9.839026e-23
1  LIQUIDBEES.NS -9.938777e-23
2       ^NSEBANK  1.000000e+00
3          ^NSEI -1.400173e-22


In [9]:
allocation_df = pd.DataFrame(

    optimized_allocations,

    index=returns.columns

)

allocation_df.to_csv(

    "../data/processed/optimized_allocations.csv"

)

print("Allocations saved.")

Allocations saved.
